In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn import metrics 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, explained_variance_score, median_absolute_error, mean_squared_log_error

In [2]:
rate_data=pd.read_csv("C:/Users/moham/Documents/ratings.csv")
movie_id_counts = rate_data['movie_id'].value_counts()
movie_id_dict = {}
for movie_id, count in movie_id_counts.items():
    if count > 480:
        movie_id_dict[movie_id] = count

movie_list=list(movie_id_dict.keys())

In [3]:
dataframes = {}
for movie_id in movie_list:
    filtered_data = rate_data[rate_data['movie_id'] == movie_id]
    dataframes[movie_id] = filtered_data

In [4]:
user_id=pd.read_csv("C:/Users/moham/Documents/user_profiles.csv")
user_id=user_id.drop("twitterName",axis=1)
merged_dataframes = {}
for movie_id in dataframes:
    new_dataframe=pd.DataFrame(dataframes[movie_id])
    merged_data = pd.merge(filtered_data, user_id, on='id')
    merged_dataframes[movie_id] = merged_data
thedatas={}
the_user_id_rated={}
for i in merged_dataframes:
    thedata=pd.DataFrame(merged_dataframes[i])
    thedata=thedata.drop("date", axis=1)
    thedata=thedata.drop("movie_id", axis=1)
    the_user_id_rated[i]=thedata["id"]
    thedata=thedata.drop("id", axis=1)
    thedata["rate"] = thedata["rate"].apply(lambda x: 1 if x > 7 else 0)
    thedatas[i] = thedata
ratedata={}
for i in thedatas: 
    rate=pd.DataFrame(thedatas[i])
    rate_y= rate["rate"]
    ratedata[i]=rate_y

    

In [5]:
lastdata={}
for i in thedatas: 
    findata=pd.DataFrame(thedatas[i])
    x=findata.drop("rate", axis=1)
    scaler = StandardScaler()
    df_standardized = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)
    lastdata[i]=df_standardized

In [6]:
lastdata

{160882:      account_creation_year  friends_count  preferred_hour  weekend_tweeter  \
 0                -1.148339      -0.350393        0.899126        -0.574304   
 1                -0.009018      -0.714261        0.753436        -0.574304   
 2                 0.560642      -0.521550        0.899126        -0.574304   
 3                -0.578679      -0.068935       -0.266393        -0.574304   
 4                 0.560642      -0.148808       -0.703463        -0.574304   
 ..                     ...            ...             ...              ...   
 374               0.560642      -0.468302        0.753436        -0.574304   
 375               0.560642      -0.652137        0.170676         1.741239   
 376               1.130302       1.163398       -0.994843        -0.574304   
 377              -1.148339      -0.005543        0.170676        -0.574304   
 378              -1.717999       0.206185       -0.703463        -0.574304   
 
      preferred_weekday  early_bird  night

In [18]:
from sklearn.metrics import accuracy_score, mean_squared_error, f1_score
metrics_DT = {}
metrics_RF = {}
for i in lastdata:
    modeldata = pd.DataFrame(lastdata[i])
    rate_data = pd.DataFrame(ratedata[i])
    x_data = modeldata
    y_data = rate_data
    X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.1, random_state=42)
    y_train = np.ravel(y_train)
    decision_tree = DecisionTreeClassifier()
    decision_tree.fit(X_train, y_train)
    decision_tree_predictions = decision_tree.predict(X_test)
    
    random_forest = RandomForestClassifier()
    random_forest.fit(X_train, y_train)
    random_forest_predictions = random_forest.predict(X_test)
    
    metrics_DT[i] = {
            'Rating': decision_tree.score(X_test, y_test),
            'Accuracy': accuracy_score(y_test, decision_tree_predictions),
            'RMSE': mean_squared_error(y_test, decision_tree_predictions, squared=False),
            'F1': f1_score(y_test, decision_tree_predictions, average='weighted')}
        
        
    metrics_RF[i] = {
            'Rating': random_forest.score(X_test, y_test),
            'Accuracy': accuracy_score(y_test, random_forest_predictions),
            'RMSE': mean_squared_error(y_test, random_forest_predictions, squared=False),
            'F1': f1_score(y_test, random_forest_predictions, average='weighted')}


In [11]:
metrics

{160882: {'DT': {'Rating': 0.7631578947368421,
   'Accuracy': 0.7631578947368421,
   'RMSE': 0.4866642633922876,
   'F1': 0.7659277727998753},
  'RF': {'Rating': 0.8508771929824561,
   'Accuracy': 0.8508771929824561,
   'RMSE': 0.38616422286061647,
   'F1': 0.7823231063440591}},
 971380: {'DT': {'Rating': 0.7368421052631579,
   'Accuracy': 0.7368421052631579,
   'RMSE': 0.512989176042577,
   'F1': 0.7479224376731303},
  'RF': {'Rating': 0.8421052631578947,
   'Accuracy': 0.8421052631578947,
   'RMSE': 0.39735970711951313,
   'F1': 0.7779448621553886}},
 867354: {'DT': {'Rating': 0.7017543859649122,
   'Accuracy': 0.7017543859649122,
   'RMSE': 0.5461186812727502,
   'F1': 0.7193635684957073},
  'RF': {'Rating': 0.8421052631578947,
   'Accuracy': 0.8421052631578947,
   'RMSE': 0.39735970711951313,
   'F1': 0.7779448621553886}},
 524439: {'DT': {'Rating': 0.7368421052631579,
   'Accuracy': 0.7368421052631579,
   'RMSE': 0.512989176042577,
   'F1': 0.7427814327485379},
  'RF': {'Rating': 

In [19]:
df_metrics_DT= pd.DataFrame.from_dict(metrics_DT, orient='index')
df_metrics_DT = df_metrics_DT.rename(columns={
    'Rating': 'DT_Rating',
    'Accuracy': 'DT_Accuracy',
    'RMSE': 'DT_RMSE',
    'F1': 'DT_F1'
})

In [20]:
df_metrics_RF= pd.DataFrame.from_dict(metrics_RF, orient='index')
df_metrics_RF = df_metrics_RF.rename(columns={
    'Rating': 'RF_Rating',
    'Accuracy': 'RF_Accuracy',
    'RMSE': 'RF_RMSE',
    'F1': 'RF_F1'
})
metrics_df=pd.concat([df_metrics_RF,df_metrics_DT],axis=1)

In [21]:
metrics_df

,RF_Rating,RF_Accuracy,RF_RMSE,RF_F1,DT_Rating,DT_Accuracy,DT_RMSE,DT_F1
160882,0.789474,0.789474,0.458831,0.696594,0.631579,0.631579,0.606977,0.611205
971380,0.789474,0.789474,0.458831,0.696594,0.631579,0.631579,0.606977,0.611205
867354,0.789474,0.789474,0.458831,0.696594,0.657895,0.657895,0.584898,0.626566
524439,0.789474,0.789474,0.458831,0.696594,0.657895,0.657895,0.584898,0.626566
929558,0.789474,0.789474,0.458831,0.696594,0.631579,0.631579,0.606977,0.611205
495280,0.789474,0.789474,0.458831,0.696594,0.657895,0.657895,0.584898,0.626566
750283,0.789474,0.789474,0.458831,0.696594,0.657895,0.657895,0.584898,0.626566
746997,0.789474,0.789474,0.458831,0.696594,0.684211,0.684211,0.561951,0.641447
575149,0.789474,0.789474,0.458831,0.696594,0.631579,0.631579,0.606977,0.611205
444796,0.815789,0.815789,0.429198,0.753775,0.684211,0.684211,0.561951,0.641447


In [22]:
predict_DT = {}
predict_RF ={}
for i in lastdata:
    modeldata = pd.DataFrame(lastdata[i])
    rate_data = pd.DataFrame(ratedata[i])
    x_data = modeldata
    y_data = rate_data
    X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=42)
    y_train = np.ravel(y_train)
    decision_tree = DecisionTreeClassifier()
    decision_tree.fit(X_train, y_train)
    decision_tree_predictions = decision_tree.predict(modeldata)
    
    random_forest = RandomForestClassifier()
    random_forest.fit(X_train, y_train)
    random_forest_predictions = random_forest.predict(modeldata)
    
    predict_DT[i] = decision_tree_predictions
    predict_RF[i] = random_forest_predictions


In [23]:
df_DT = pd.DataFrame.from_dict(predict_DT, orient='index')
df_RF = pd.DataFrame.from_dict(predict_RF, orient='index')
transposed_df1 = df_DT.T
transposed_df2 = df_RF.T
transposed_df1.to_csv("nofuzzy_DT_pre.csv",index= False)
transposed_df2.to_csv("nofuzzy_RF_pre.csv",index= False)
metrics_df.to_csv("nofuzzy_metrics_df.csv",index= False)